# Test pipeline trên trận mới: video → luồng YES → luồng NO (Colab, v4.7)

Chạy **hệ thống đã train** trên trận đấu mới (không có nhãn, không train gì):

| Bước | Việc làm |
|---|---|
| 1–4 | GPU, cấu hình, mount Drive, lấy code |
| 5 | Video 2 hiệp → npy TimeSformer 768 chiều |
| 6 | Phép ép 768 → **npy 8576 chiều** (phép ép đã học ở notebook train) |
| 7 | **Luồng YES**: model đã train bắt sự kiện + sinh câu bình luận → `results_dense_captioning.json` |
| 8 | **Luồng NO**: tìm khoảng trống giữa các sự kiện YES, LLM ElevenLabs nói thông tin từ metadata FIFA |
| 9 | Xem timeline YES + NO |

**Cần có trên Drive** (`MyDrive/soccer_project/`), **để Private**:
- `video_test/<giải>/<mùa>/<trận>/1_224p.mp4` và `2_224p.mp4`: hai hiệp đã cắt bằng `cut_halves.py`.
- `video_test/metadata/<trận>/`: kết quả `fifa_crawl.py` (`facts.csv`, `events.csv`, `match_info.csv`...). Tên thư mục trùng tên thư mục trận.
- Từ notebook train: `sn-models/<MODEL_NAME>/` (checkpoint `spotting/` và `caption/`), file phép ép trong `sn-models/`, weights TimeSformer `*.pdparams`.

**API key ElevenLabs**: đặt trong Colab Secrets (biểu tượng 🔑 bên trái) tên `ELEVENLABS_API_KEY`, bật quyền cho notebook. Không dán key vào cell. Không có key thì bước 8 chạy chế độ `template` (câu fact gốc, không LLM).

Chọn GPU: `Runtime → Change runtime type → T4 GPU`.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Cấu hình

- `MODEL_NAME`, `MAPPER_FILE`: model và phép ép **đi theo cặp**, lấy đúng cặp đã train (notebook train cell 2).
- `NMS_THRESHOLD`, `TOP_K`: tổ hợp tốt nhất mà notebook train cell 12 chọn trên valid.

In [ ]:
GITHUB_REPO = "https://github.com/ThuLinh3009/soccer_project.git"
GIT_BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/soccer_project"
WORK_DIR = "/content/soccernet-caption-deltalab"   # code
TEST_DATA = "/content/test_data"                   # npy của các trận test
TEST_DIR_NAME = "video_test"                       # thư mục trên Drive chứa video + metadata trận test

MODEL_NAME = "gpt2-video60-mlp"                    # model đã train (sn-models/<MODEL_NAME>)
MAPPER_FILE = "mapper_timesformer_mlp_c4.pt"       # phép ép đã học cùng model đó
FEATURES = "baidu_from_timesformer.npy"
GPT_TYPE = "gpt2"
NMS_THRESHOLD = 0.5                                # từ notebook train cell 12
TOP_K = 3
TS_BATCH = 32

LLM = "elevenlabs"                                 # "template": không gọi LLM, dùng câu fact gốc
LLM_MODEL = "gemini-2.5-flash"

## 3. Mount Google Drive, tìm trận test, model và weights

In [ ]:
from google.colab import drive
import os, glob
drive.mount("/content/drive", force_remount=True)

if not os.path.isdir(DRIVE_ROOT):
    found = [p for p in glob.glob("/content/drive/MyDrive/**/soccer_project", recursive=True)
             + glob.glob("/content/drive/Shareddrives/*/soccer_project") if os.path.isdir(p)]
    assert found, f"Không thấy {DRIVE_ROOT}: sửa DRIVE_ROOT ở cell 2"
    DRIVE_ROOT = found[0]

TEST_SRC = f"{DRIVE_ROOT}/{TEST_DIR_NAME}"
DRIVE_MODEL_DIR = f"{DRIVE_ROOT}/sn-models"
MAPPER = f"{DRIVE_MODEL_DIR}/{MAPPER_FILE}"
FEAT_CACHE = f"{TEST_SRC}/features"                # npy 768 chiều lưu lại trên Drive
OUT_YES = f"{TEST_SRC}/output/yes"
OUT_TIMELINE = f"{TEST_SRC}/output/timeline"
weights = sorted(glob.glob(f"{DRIVE_ROOT}/**/*.pdparams", recursive=True))
TS_WEIGHTS = weights[0] if weights else None

videos = sorted(glob.glob(f"{TEST_SRC}/*/*/*/[12]_224p.mp4"))
GAMES = sorted({os.path.relpath(os.path.dirname(v), TEST_SRC) for v in videos})
print("DRIVE_ROOT:", DRIVE_ROOT)
for g in GAMES:
    halves = sorted(os.path.basename(v)[0] for v in videos if os.path.dirname(v).endswith(g))
    meta = f"{TEST_SRC}/metadata/{os.path.basename(g)}/facts.csv"
    print(f"{'✅' if halves == ['1', '2'] else '⚠️ thiếu hiệp'} hiệp {halves} | metadata {'✅' if os.path.exists(meta) else '❌'} | {g}")
print("model   :", MODEL_NAME, "✅" if os.path.exists(f"{DRIVE_MODEL_DIR}/{MODEL_NAME}/caption/model.pth.tar") else "❌ không thấy checkpoint")
print("phép ép :", MAPPER, "✅" if os.path.exists(MAPPER) else "❌")
print("weights :", TS_WEIGHTS or "❌ KHÔNG THẤY file .pdparams")
assert GAMES and os.path.exists(MAPPER) and TS_WEIGHTS
assert os.path.exists(f"{DRIVE_MODEL_DIR}/{MODEL_NAME}/spotting/model.pth.tar")

## 4. Lấy code từ GitHub và cài thư viện

In [ ]:
if os.path.isdir(f"{WORK_DIR}/.git"):
    !git -C {WORK_DIR} fetch -q origin && git -C {WORK_DIR} reset -q --hard origin/{GIT_BRANCH}
else:
    !git clone -q -b {GIT_BRANCH} {GITHUB_REPO} {WORK_DIR}
%cd {WORK_DIR}
!git log --oneline -1
!grep -m1 "^## v" CHANGELOG.md
!pip install -q SoccerNet av tiktoken transformers wandb pycocoevalcap psutil six websockets
!pip install -q --no-deps git+https://github.com/Maluuba/nlg-eval.git@master

import subprocess
os.environ["WANDB_MODE"] = "offline"

def run(cmd):
    # chạy lệnh, in log trực tiếp; báo lỗi ngay tại cell này nếu lệnh thất bại
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=WORK_DIR)
    for line in p.stdout:
        if "it/s]" in line or "s/it]" in line:
            continue
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Lệnh lỗi (exit {p.returncode}): xem log ngay phía trên")

# models/ trỏ thẳng tới sn-models trên Drive (checkpoint đã train)
if not os.path.islink("models"):
    !rm -rf models
!ln -sfn "{DRIVE_MODEL_DIR}" models

## 5. Video → npy TimeSformer 768 chiều

Mỗi hiệp mất khoảng 5–10 phút trên T4. Kết quả lưu lại trên Drive (`video_test/features/`), chạy lại thì không trích lại.

In [ ]:
import shutil
def sync(src_root, dst_root):
    for f in glob.glob(f"{src_root}/**/*_timesformer768.npy", recursive=True):
        dst = os.path.join(dst_root, os.path.relpath(f, src_root))
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copyfile(f, dst)

if os.path.isdir(FEAT_CACHE):
    sync(FEAT_CACHE, TEST_DATA)
try:
    run(["python", "timesformer_extract.py", "--weights", TS_WEIGHTS, "--video_root", TEST_SRC,
         "--video_name", "{half}_224p.mp4", "--out_root", TEST_DATA, "--batch_size", str(TS_BATCH), "--games", *GAMES])
finally:
    sync(TEST_DATA, FEAT_CACHE)
missing = [f"{g}/{h}" for g in GAMES for h in (1, 2) if not os.path.exists(f"{TEST_DATA}/{g}/{h}_timesformer768.npy")]
assert not missing, f"Còn thiếu: {missing}. Chạy lại cell này"

## 6. Phép ép 768 → npy 8576 chiều

Dùng **đúng phép ép đã học** ở notebook train (không học lại). Mỗi hiệp ra `1_/2_baidu_from_timesformer.npy`, kích thước (số giây, 8576).

In [ ]:
run(["python", "feature_mapper.py", "apply", "--soccernet_path", TEST_DATA, "--mapper", MAPPER,
     "--games", *GAMES, "--out_name", FEATURES, "--overwrite"])
import numpy as np
for g in GAMES:
    print([np.load(f"{TEST_DATA}/{g}/{h}_{FEATURES}", mmap_mode="r").shape for h in (1, 2)], g)

## 7. Luồng YES: bắt sự kiện + sinh câu bình luận

Model đã train (`spotting` + `caption`) chạy trên npy 8576 chiều của trận mới: tìm thời điểm sự kiện (NMS), rồi GPT-2 sinh một câu cho mỗi sự kiện. Không có nhãn nên không chấm điểm.
Kết quả: `video_test/output/yes/<trận>/results_dense_captioning.json`.

In [ ]:
run(["python", "infer_match.py", "--model_name", MODEL_NAME, "--features", FEATURES, "--gpt_type", GPT_TYPE,
     "--nms_threshold", str(NMS_THRESHOLD), "--top_k", str(TOP_K), "--out_dir", OUT_YES,
     "--game_dirs", *[f"{TEST_DATA}/{g}" for g in GAMES]])

## 8. Luồng NO: LLM ElevenLabs trong các khoảng trống

- Khoảng trống = khoảng giữa hai sự kiện **của luồng YES ở bước 7**.
- Mỗi khoảng trống chọn một fact trong `facts.csv` (metadata FIFA), LLM viết lại thành câu nói. Câu có con số không có trong ngữ cảnh thì dùng câu fact gốc.
- Không train, không cần nhãn.

In [ ]:
from google.colab import userdata
llm = LLM
try:
    os.environ["ELEVENLABS_API_KEY"] = userdata.get("ELEVENLABS_API_KEY")
except Exception as e:
    print("⚠️ không đọc được secret ELEVENLABS_API_KEY:", type(e).__name__, "→ chạy chế độ template")
    llm = "template"

if llm == "elevenlabs":
    # kiểm tra key trước: 401 = key sai / đã xoá / thiếu quyền ElevenAgents
    import requests
    key = os.environ["ELEVENLABS_API_KEY"].strip().strip("\"'").strip()
    os.environ["ELEVENLABS_API_KEY"] = key
    r = requests.get("https://api.elevenlabs.io/v1/convai/agents", params={"page_size": 1},
                     headers={"xi-api-key": key}, timeout=30)
    print(f"kiểm tra key ({len(key)} ký tự, bắt đầu '{key[:3]}'):", r.status_code, "OK" if r.ok else r.text[:300])
    assert r.ok, ("ElevenLabs không nhận key: tạo key mới (bật quyền ElevenAgents), dán lại vào secret "
                  "ELEVENLABS_API_KEY, rồi chạy lại cell này")

for g in GAMES:
    name = os.path.basename(g)
    run(["python", "no_flow_llm.py", "--metadata", f"{TEST_SRC}/metadata/{name}", "--video_dir", f"{TEST_SRC}/{g}",
         "--yes_events", f"{OUT_YES}/{name}/results_dense_captioning.json", "--llm", llm, "--llm_model", LLM_MODEL,
         "--agent_cache", f"{DRIVE_MODEL_DIR}/elevenlabs_agent.json", "--out_dir", OUT_TIMELINE])

## 9. Timeline YES + NO

`[EVENT]`: câu của luồng YES (model). `[INFO ]`: câu của luồng NO (LLM từ metadata).

In [ ]:
for g in GAMES:
    path = f"{OUT_TIMELINE}/{os.path.basename(g)}/commentary_timeline.txt"
    lines = open(path, encoding="utf-8").read().splitlines()
    print(f"\n===== {g}: {sum('[EVENT]' in l for l in lines)} câu YES, {sum('[INFO ]' in l for l in lines)} câu NO")
    print("\n".join(lines[:60]))